[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vu-topics-in-big-data-2026/examples/blob/main/example-kubernetes/notebook.ipynb)

# IoT Temperature Monitoring: Docker & Kubernetes Demo

This notebook walks through building and deploying a real-time IoT anomaly detection pipeline using **Docker Compose** and **Kubernetes (minikube)**.

**The pipeline:**
```
┌──────────────┐       ┌──────────┐       ┌──────────────┐
│   Producer   │──────>│ Redpanda │──────>│   Analyzer   │
│ reads CSV,   │       │ (Kafka   │       │ flags temps   │
│ sends 1/sec  │       │  broker) │       │ > 40C or < 5C │
└──────────────┘       └──────────┘       └──────────────┘
```

- **Redpanda** — Kafka-compatible message broker (lightweight, no JVM)
- **Producer** — Python script that reads sensor CSV and streams one reading/sec
- **Analyzer** — Python script that consumes readings and flags temperature anomalies

---
## 1. Setup: Install Docker, minikube, and clone the repo

Colab runs on a Linux VM, so we can install Docker and minikube directly. This cell installs everything we need.

In [ ]:
%%bash
# Install Docker
apt-get update -qq
apt-get install -y -qq docker.io docker-compose-v2 > /dev/null 2>&1
dockerd --iptables=false &> /dev/null &
sleep 5
docker --version
docker compose version

In [ ]:
%%bash
# Install minikube and kubectl
curl -sLo /usr/local/bin/minikube https://storage.googleapis.com/minikube/releases/latest/minikube-linux-amd64
chmod +x /usr/local/bin/minikube

curl -sLo /usr/local/bin/kubectl "https://dl.k8s.io/release/$(curl -sL https://dl.k8s.io/release/stable.txt)/bin/linux/amd64/kubectl"
chmod +x /usr/local/bin/kubectl

minikube version
kubectl version --client

In [ ]:
%%bash
# Clone the repo and navigate to the demo directory
if [ ! -d "examples" ]; then
    git clone https://github.com/vu-topics-in-big-data-2026/examples.git
fi
ls examples/example-kubernetes/

In [ ]:
import os
os.chdir('examples/example-kubernetes')
print('Working directory:', os.getcwd())

---
## 2. Explore the data

Before running anything, let's look at the sensor data our pipeline will process. The CSV has 200 readings from 5 IoT sensors, with ~10% anomalies injected (temperatures below 5C or above 40C).

In [ ]:
import pandas as pd

df = pd.read_csv('data/sensors.csv')
print(f'Total readings: {len(df)}')
print(f'Sensors: {df["sensor_id"].unique()}')
print(f'Locations: {df["location"].unique()}')
print(f'\nTemperature range: {df["temperature_c"].min():.1f}C to {df["temperature_c"].max():.1f}C')
print(f'Anomalies (< 5C or > 40C): {len(df[(df["temperature_c"] < 5) | (df["temperature_c"] > 40)])} / {len(df)}')
df.head(10)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 4))
for sensor_id, group in df.groupby('sensor_id'):
    ax.plot(group.index, group['temperature_c'], label=sensor_id, alpha=0.7)

ax.axhline(y=40, color='red', linestyle='--', alpha=0.5, label='High threshold (40C)')
ax.axhline(y=5, color='blue', linestyle='--', alpha=0.5, label='Low threshold (5C)')
ax.set_xlabel('Reading index')
ax.set_ylabel('Temperature (C)')
ax.set_title('Sensor Readings with Anomaly Thresholds')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

---
## 3. Inspect the code

Let's look at the key files before building and running anything.

### Producer
Reads the CSV, connects to Redpanda (with retries), and sends one reading per second as JSON.

In [ ]:
!cat producer/producer.py

### Analyzer
Consumes readings from the topic and flags temperatures above 40C or below 5C.

In [ ]:
!cat analyzer/analyzer.py

### Dockerfiles
Both use the same pattern: `python:3.11-slim` base, copy requirements first (for layer caching), then copy the app code.

In [ ]:
print('=== Producer Dockerfile ===')
!cat producer/Dockerfile
print('\n=== Analyzer Dockerfile ===')
!cat analyzer/Dockerfile

### compose.yaml
Defines all three services (Redpanda, producer, analyzer) with health checks and startup ordering.

In [ ]:
!cat compose.yaml

---
## 4. Part A: Docker Compose

Docker Compose runs all three containers on a single machine with shared networking. It handles startup order via health checks — the producer and analyzer wait for Redpanda to be healthy before starting.

### Step 1: Build the images

Each `docker build` packages the Python app and its dependencies into a container image.

In [ ]:
!docker build -t iot-producer:v1 ./producer
!docker build -t iot-analyzer:v1 ./analyzer

### Step 2: Start the pipeline

`docker compose up -d` starts all services in the background. Compose creates an isolated network so containers can find each other by name (e.g., the producer connects to `redpanda:9092`).

In [ ]:
!docker compose up -d

In [ ]:
# Verify all three containers are running
!docker compose ps

### Step 3: Watch the output

Give it a few seconds for the producer to start streaming, then check the analyzer logs. You should see `OK` for normal readings and `ANOMALY HIGH` / `ANOMALY LOW` for readings outside the 5-40C range.

In [ ]:
import time
print('Waiting 15 seconds for data to flow...')
time.sleep(15)

In [ ]:
# Check producer logs — should show "Sent: sensor-XX | location | temp"
!docker compose logs producer --tail 10

In [ ]:
# Check analyzer logs — should show OK / ANOMALY HIGH / ANOMALY LOW
!docker compose logs analyzer --tail 15

### Step 4: Clean up

This stops all containers, removes them, and tears down the network.

In [ ]:
!docker compose down

---
## 5. Part B: Kubernetes (minikube)

Now we deploy the **exact same pipeline** to a Kubernetes cluster. Instead of one `compose.yaml`, we use separate YAML manifests that declare the desired state. Kubernetes continuously works to make reality match — this is what gives us self-healing and scaling.

### Why Kubernetes over Compose?

| | Docker Compose | Kubernetes |
|---|---|---|
| **What happens on crash** | Container stays dead | Auto-replaced in seconds |
| **Scaling** | Not supported | One command: `kubectl scale` |
| **Best for** | Local dev and testing | Staging and production |

### Step 1: Start minikube

This boots a single-node Kubernetes cluster inside a Docker container on this Colab VM.

In [ ]:
!minikube start --driver=docker --force

### Step 2: Load images into minikube

minikube runs its own Docker daemon. We need to copy our locally built images into it so Kubernetes can use them.

In [ ]:
!minikube image load iot-producer:v1
!minikube image load iot-analyzer:v1

### Step 3: Deploy the pipeline

Each `kubectl apply` tells Kubernetes "make this resource exist." We apply them in order so that Redpanda is ready before the producer and analyzer start.

Let's first look at what resources we're creating:

In [ ]:
# Inspect the Kubernetes manifests
!echo '=== Namespace ===' && cat k8s/namespace.yaml
!echo '\n=== ConfigMap ===' && cat k8s/configmap.yaml

In [ ]:
!echo '=== Redpanda StatefulSet ===' && cat k8s/redpanda-statefulset.yaml

In [ ]:
!echo '=== Producer Deployment ===' && cat k8s/producer-deployment.yaml
!echo '\n=== Analyzer Deployment ===' && cat k8s/analyzer-deployment.yaml

Now apply them:

In [ ]:
# Create namespace and config
!kubectl apply -f k8s/namespace.yaml
!kubectl -n streaming-iot create configmap sensor-data --from-file=data/sensors.csv
!kubectl apply -f k8s/configmap.yaml

In [ ]:
# Deploy Redpanda and wait for it to be ready
!kubectl apply -f k8s/redpanda-statefulset.yaml
!kubectl apply -f k8s/redpanda-service.yaml
!kubectl -n streaming-iot wait --for=condition=Ready pod/redpanda-0 --timeout=120s

In [ ]:
# Deploy producer and analyzer
!kubectl apply -f k8s/producer-deployment.yaml
!kubectl apply -f k8s/analyzer-deployment.yaml
!kubectl apply -f k8s/analyzer-service.yaml

### Step 4: Watch the logs

Give the pods a moment to start, then check the output. This is the K8s equivalent of `docker compose logs`.

In [ ]:
# Check all pods are running
!kubectl -n streaming-iot get pods

In [ ]:
import time
print('Waiting 20 seconds for data to flow...')
time.sleep(20)

In [ ]:
# Check analyzer logs — same output as Docker Compose
!kubectl -n streaming-iot logs deployment/analyzer --tail 15

---
### Step 5: Scaling and load balancing

This is where Kubernetes shines over Docker Compose. The topic has **3 partitions** keyed by `sensor_id`. When we scale to 3 analyzer replicas, Kafka assigns one partition per analyzer — true parallel processing, with a single command.

In [ ]:
# Scale from 1 analyzer to 3
!kubectl -n streaming-iot scale deployment analyzer --replicas=3

In [ ]:
import time
print('Waiting 15 seconds for new pods to start...')
time.sleep(15)

# All 3 pods should be Running
!kubectl -n streaming-iot get pods -l app=analyzer

In [ ]:
import time
time.sleep(10)

# Check logs from each analyzer pod — each should handle different sensors
import subprocess
result = subprocess.run(
    ['kubectl', '-n', 'streaming-iot', 'get', 'pods', '-l', 'app=analyzer',
     '-o', 'jsonpath={.items[*].metadata.name}'],
    capture_output=True, text=True
)
pods = result.stdout.strip().split()
for pod in pods:
    print(f'\n=== {pod} ===')
    !kubectl -n streaming-iot logs {pod} --tail 5

Each pod processes a **different subset** of sensors. This is Kafka consumer group rebalancing — each partition is assigned to exactly one consumer in the group.

---
### Step 6: Self-healing

In production, containers crash — out-of-memory errors, bugs, hardware failures. Kubernetes detects this and automatically replaces dead pods. Let's simulate a crash by deleting a pod and watching K8s replace it.

In [ ]:
# See the current pods and their ages
print('BEFORE killing a pod:')
!kubectl -n streaming-iot get pods -l app=analyzer

In [ ]:
# Kill one pod (simulates a crash)
import subprocess
result = subprocess.run(
    ['kubectl', '-n', 'streaming-iot', 'get', 'pods', '-l', 'app=analyzer',
     '-o', 'jsonpath={.items[0].metadata.name}'],
    capture_output=True, text=True
)
pod_to_kill = result.stdout.strip()
print(f'Killing pod: {pod_to_kill}')
!kubectl -n streaming-iot delete pod {pod_to_kill} --wait=false

In [ ]:
import time
print('Waiting 10 seconds for K8s to replace the pod...')
time.sleep(10)

print('AFTER self-healing:')
!kubectl -n streaming-iot get pods -l app=analyzer

Notice that one pod has a much younger `AGE` — that's the replacement. The Deployment controller noticed the actual state (2 pods) didn't match the desired state (3 pods) and immediately created a new one. This "observe, diff, act" loop is how Kubernetes maintains your system 24/7 without human intervention.

---
### Step 7: Clean up

In [ ]:
# Delete the namespace (removes all resources inside it)
!kubectl delete namespace streaming-iot

# Stop minikube
!minikube stop

---
## Summary

We deployed the same IoT anomaly detection pipeline in two ways:

1. **Docker Compose** — one command (`docker compose up`), great for local dev, but no self-healing or scaling
2. **Kubernetes** — more setup (YAML manifests, `kubectl apply`), but gives us:
   - **Scaling**: `kubectl scale` to add more analyzer replicas
   - **Self-healing**: killed pods are automatically replaced
   - **Declarative state**: you describe *what* you want, K8s figures out *how*

In production, teams use Kubernetes (or managed versions like AWS EKS, Google GKE, Azure AKS) to run streaming pipelines, web services, ML inference, and more — all with the same deploy/scale/heal patterns you just practiced.